# Abstract Factory (Python)

**Đối tượng:** người đã biết class/function cơ bản và muốn học Design Patterns.

**Yêu cầu trước:**
- Python OOP cơ bản
- Khái niệm interface/abstract class (ABC)

**Mục tiêu:**
- Hiểu Abstract Factory dùng khi nào
- Tự viết được ví dụ tạo *một họ* sản phẩm liên quan (Button + Checkbox)
- Biết cách đổi “biến thể” (Windows/Mac) mà không sửa code phía client


## Outline

1. Bài toán (vì sao cần Abstract Factory)
2. Định nghĩa sản phẩm (Product interfaces)
3. Định nghĩa Factory trừu tượng + các Factory cụ thể
4. Client sử dụng Factory
5. Ghi chú: khi dùng / khi không nên dùng
6. Bài tập


In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod


## 1) Bài toán

Giả sử bạn đang viết UI toolkit hỗ trợ nhiều “biến thể” (Windows, macOS).

Mỗi biến thể cần một **họ sản phẩm** tương thích:
- `Button`
- `Checkbox`

Yêu cầu:
- Nếu chọn biến thể `Windows` thì cả Button và Checkbox đều phải là phiên bản Windows.
- Client code (Application) **không** phụ thuộc vào các lớp cụ thể như `WindowsButton`.

Đây là điểm mạnh của **Abstract Factory**: tạo *một họ* đối tượng liên quan mà không lộ lớp cụ thể.


## 2) Product interfaces (các sản phẩm trừu tượng)


In [ ]:
class Button(ABC):
    @abstractmethod
    def render(self) -> str:
        raise NotImplementedError


class Checkbox(ABC):
    @abstractmethod
    def render(self) -> str:
        raise NotImplementedError


## 3) Concrete products (các sản phẩm cụ thể theo từng biến thể)


In [ ]:
class WindowsButton(Button):
    def render(self) -> str:
        return "[Windows] Button"


class MacButton(Button):
    def render(self) -> str:
        return "[macOS] Button"


class WindowsCheckbox(Checkbox):
    def render(self) -> str:
        return "[Windows] Checkbox"


class MacCheckbox(Checkbox):
    def render(self) -> str:
        return "[macOS] Checkbox"


## 4) Abstract Factory + Concrete Factories


In [ ]:
class UIFactory(ABC):
    @abstractmethod
    def create_button(self) -> Button:
        raise NotImplementedError

    @abstractmethod
    def create_checkbox(self) -> Checkbox:
        raise NotImplementedError


class WindowsUIFactory(UIFactory):
    def create_button(self) -> Button:
        return WindowsButton()

    def create_checkbox(self) -> Checkbox:
        return WindowsCheckbox()


class MacUIFactory(UIFactory):
    def create_button(self) -> Button:
        return MacButton()

    def create_checkbox(self) -> Checkbox:
        return MacCheckbox()


## 5) Client code: chỉ phụ thuộc vào interface

`Application` nhận vào một `UIFactory`. Khi cần Button/Checkbox, nó gọi factory.

Điểm quan trọng: `Application` không cần biết (và không import) `WindowsButton`, `MacButton`, ...


In [ ]:
class Application:
    def __init__(self, factory: UIFactory) -> None:
        self._factory = factory

    def paint(self) -> list[str]:
        button = self._factory.create_button()
        checkbox = self._factory.create_checkbox()
        return [button.render(), checkbox.render()]


### Demo: đổi biến thể mà không đổi code phía client


In [ ]:
def run_app(os_name: str) -> list[str]:
    factory: UIFactory
    if os_name.lower() in {"win", "windows"}:
        factory = WindowsUIFactory()
    elif os_name.lower() in {"mac", "macos", "osx"}:
        factory = MacUIFactory()
    else:
        raise ValueError(f"Unsupported OS: {os_name}")

    return Application(factory).paint()


run_app("windows"), run_app("mac")


## 6) Ghi chú nhanh

**Nên dùng khi:**
- Có nhiều *biến thể* (Windows/macOS, Light/Dark, MySQL/Postgres...)
- Mỗi biến thể gồm nhiều sản phẩm **đi theo bộ** và cần tương thích

**Cân nhắc khi:**
- Chỉ có 1 loại sản phẩm cần tạo → thường `Factory Method`/constructor là đủ
- Số “họ sản phẩm” tăng nhanh → có thể làm hệ thống nhiều class hơn cần thiết

Thực tế, bạn thường kết hợp Abstract Factory với **Dependency Injection** (truyền factory từ ngoài vào).


## 7) Bài tập

1. Thêm họ sản phẩm thứ 3: `Window` (hoặc `Menu`) cho cả Windows/macOS.
2. Thêm biến thể thứ 3: `Linux`.
3. Sửa `Application` để nhận `factory` từ cấu hình (dictionary) thay vì if/else.
